<a href="https://colab.research.google.com/github/namiri/task/blob/test/finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator



In [5]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3)) #MobileNetV2 with pre-trained weights. include_top=False to exclude the model's default classification layers.
# Add Custom Layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)  # 1024 units as an example
predictions = Dense(2, activation='softmax')(x)  # num_classes = number of classes in our dataset

model = Model(inputs=base_model.input, outputs=predictions)


In [6]:
for layer in base_model.layers: #Freeze the Base Model Layers
    layer.trainable = False


In [7]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])


In [14]:
train_datagen = ImageDataGenerator(rescale=1.0/255)
train_generator = train_datagen.flow_from_directory(
    '/content/drive/MyDrive/test/source',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

val_datagen = ImageDataGenerator(rescale=1.0/255)
val_generator = val_datagen.flow_from_directory(
    '/content/drive/MyDrive/test/target',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)


Found 0 images belonging to 0 classes.
Found 0 images belonging to 0 classes.


In [17]:
model.fit(train_generator,
          epochs=5,
          validation_data=val_generator)

for layer in base_model.layers[-20:]:  # Unfreeze the last 20 layers
    layer.trainable = True

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.fit(train_generator,
          epochs=5,
          validation_data=val_generator)


ValueError: Must provide at least one structure